# Advanced Retrieval with LangChain

In the following notebook, we'll explore various methods of advanced retrieval using LangChain!

We'll touch on:

- Naive Retrieval
- Best-Matching 25 (BM25)
- Multi-Query Retrieval
- Parent-Document Retrieval
- Contextual Compression (a.k.a. Rerank)
- Ensemble Retrieval
- Semantic chunking

We'll also discuss how these methods impact performance on our set of documents with a simple RAG chain.

There will be two breakout rooms:

- 🤝 Breakout Room Part #1
  - Task 1: Getting Dependencies!
  - Task 2: Data Collection and Preparation
  - Task 3: Setting Up QDrant!
  - Task 4-10: Retrieval Strategies
- 🤝 Breakout Room Part #2
  - Activity: Evaluate with Ragas

# 🤝 Breakout Room Part #1

## Task 1: Getting Dependencies!

We're going to need a few specific LangChain community packages, like OpenAI (for our [LLM](https://platform.openai.com/docs/models) and [Embedding Model](https://platform.openai.com/docs/guides/embeddings)) and Cohere (for our [Reranker](https://cohere.com/rerank)).

We'll also provide our OpenAI key, as well as our Cohere API key.

In [58]:
### API key management

### Reminder: Place .env file inside the root of the project folder so when calling the below from inside the notebook it should find the .env fule and load it inside the notebook environment
### PLEASE ADD THIS `.env` FILE TO YOUR PROJECT'S `.gitignore` file before committing and pushing the changes to your remote repo, as it contains API Keys and Secrets in it

import os
from dotenv import load_dotenv

load_dotenv(dotenv_path="../.env")

print("OPENAI_API_KEY" in os.environ)
print("LANGCHAIN_API_KEY" in os.environ)
print("TAVILY_API_KEY" in os.environ)


True
True
True


In [59]:
#import os
#import getpass

#os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API Key:")

In [60]:
#os.environ["COHERE_API_KEY"] = getpass.getpass("Cohere API Key:")

## Task 2: Data Collection and Preparation

We'll be using our Loan Data once again - this time the strutured data available through the CSV!

### Data Preparation

We want to make sure all our documents have the relevant metadata for the various retrieval strategies we're going to be applying today.

In [61]:
from langchain_community.document_loaders.csv_loader import CSVLoader
from datetime import datetime, timedelta

loader = CSVLoader(
    file_path=f"./data/complaints.csv",
    metadata_columns=[
      "Date received", 
      "Product", 
      "Sub-product", 
      "Issue", 
      "Sub-issue", 
      "Consumer complaint narrative", 
      "Company public response", 
      "Company", 
      "State", 
      "ZIP code", 
      "Tags", 
      "Consumer consent provided?", 
      "Submitted via", 
      "Date sent to company", 
      "Company response to consumer", 
      "Timely response?", 
      "Consumer disputed?", 
      "Complaint ID"
    ]
)

loan_complaint_data = loader.load()

for doc in loan_complaint_data:
    doc.page_content = doc.metadata["Consumer complaint narrative"]

Let's look at an example document to see if everything worked as expected!

In [62]:
loan_complaint_data[0]

Document(metadata={'source': './data/complaints.csv', 'row': 0, 'Date received': '03/27/25', 'Product': 'Student loan', 'Sub-product': 'Federal student loan servicing', 'Issue': 'Dealing with your lender or servicer', 'Sub-issue': 'Trouble with how payments are being handled', 'Consumer complaint narrative': "The federal student loan COVID-19 forbearance program ended in XX/XX/XXXX. However, payments were not re-amortized on my federal student loans currently serviced by Nelnet until very recently. The new payment amount that is effective starting with the XX/XX/XXXX payment will nearly double my payment from {$180.00} per month to {$360.00} per month. I'm fortunate that my current financial position allows me to be able to handle the increased payment amount, but I am sure there are likely many borrowers who are not in the same position. The re-amortization should have occurred once the forbearance ended to reduce the impact to borrowers.", 'Company public response': 'None', 'Company'

## Task 3: Setting up QDrant!

Now that we have our documents, let's create a QDrant VectorStore with the collection name "LoanComplaints".

We'll leverage OpenAI's [`text-embedding-3-small`](https://openai.com/blog/new-embedding-models-and-api-updates) because it's a very powerful (and low-cost) embedding model.

> NOTE: We'll be creating additional vectorstores where necessary, but this pattern is still extremely useful.

In [63]:
from langchain_community.vectorstores import Qdrant
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = Qdrant.from_documents(
    loan_complaint_data,
    embeddings,
    location=":memory:",
    collection_name="LoanComplaints"
)

## Task 4: Naive RAG Chain

Since we're focusing on the "R" in RAG today - we'll create our Retriever first.

### R - Retrieval

This naive retriever will simply look at each review as a document, and use cosine-similarity to fetch the 10 most relevant documents.

> NOTE: We're choosing `10` as our `k` here to provide enough documents for our reranking process later

In [64]:
naive_retriever = vectorstore.as_retriever(search_kwargs={"k" : 10})

### A - Augmented

We're going to go with a standard prompt for our simple RAG chain today! Nothing fancy here, we want this to mostly be about the Retrieval process.

In [65]:
from langchain_core.prompts import ChatPromptTemplate

RAG_TEMPLATE = """\
You are a helpful and kind assistant. Use the context provided below to answer the question.

If you do not know the answer, or are unsure, say you don't know.

Query:
{question}

Context:
{context}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_TEMPLATE)

### G - Generation

We're going to leverage `gpt-4.1-nano` as our LLM today, as - again - we want this to largely be about the Retrieval process.

In [66]:
from langchain_openai import ChatOpenAI

chat_model = ChatOpenAI(model="gpt-4.1-nano")

### LCEL RAG Chain

We're going to use LCEL to construct our chain.

> NOTE: This chain will be exactly the same across the various examples with the exception of our Retriever!

In [67]:
from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser

naive_retrieval_chain = (
    # INVOKE CHAIN WITH: {"question" : "<<SOME USER QUESTION>>"}
    # "question" : populated by getting the value of the "question" key
    # "context"  : populated by getting the value of the "question" key and chaining it into the base_retriever
    {"context": itemgetter("question") | naive_retriever, "question": itemgetter("question")}
    # "context"  : is assigned to a RunnablePassthrough object (will not be called or considered in the next step)
    #              by getting the value of the "context" key from the previous step
    | RunnablePassthrough.assign(context=itemgetter("context"))
    # "response" : the "context" and "question" values are used to format our prompt object and then piped
    #              into the LLM and stored in a key called "response"
    # "context"  : populated by getting the value of the "context" key from the previous step
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's see how this simple chain does on a few different prompts.

> NOTE: You might think that we've cherry picked prompts that showcase the individual skill of each of the retrieval strategies - you'd be correct!

In [68]:
naive_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'Based on the provided context, the most common issue with loans appears to be related to problems with the handling and management of student loans, including errors in loan balances, misapplied payments, wrongful denials of payment plans, incorrect information on credit reports, and difficulties in repayment or loan forgiveness. \n\nMany complaints revolve around mismanagement, lack of transparency, improper transfer of loans without notice, and disputes over interest and repayment terms. These recurring problems suggest that issues with loan servicing and administration are among the most common concerns reported by consumers.'

In [69]:
naive_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided complaints, yes, some complaints were not handled in a timely manner. Specifically, there is at least one complaint where the response was marked as "No" for timely response:\n\n- Complaint ID: 12709087 (submitted to MOHELA on 03/28/25) noted that the company response to the consumer was "Closed with explanation," and the response time was "No," indicating it was not handled within a timely manner.\n\nAdditionally, multiple complaints mention prolonged unresolved issues and delays in responses or actions, with some complaints having been ongoing for over a year without resolution.'

In [70]:
naive_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

"People failed to pay back their loans often due to a combination of factors highlighted in the complaints:\n\n1. **Lack of clear communication and notification:** Borrowers were frequently not informed about key changes such as loan transfers between servicers, the start of repayment, or updates regarding payment resumption, leading to missed payments and delinquency.\n\n2. **Ineffective repayment options:** The available options like forbearance or deferment often resulted in interest continuing to accrue, making the total debt grow over time and difficult to manage.\n\n3. **Financial hardships:** Many borrowers faced financial difficulties, stagnating wages, high living costs, or unemployment, which made it impossible to keep up with payments, especially when payment plans were extended or complex.\n\n4. **Mismanagement and lack of transparency:** Complaints reveal that borrowers often received inconsistent or confusing information about their balances, interest calculations, and pa

Overall, this is not bad! Let's see if we can make it better!

## Task 5: Best-Matching 25 (BM25) Retriever

Taking a step back in time - [BM25](https://www.nowpublishers.com/article/Details/INR-019) is based on [Bag-Of-Words](https://en.wikipedia.org/wiki/Bag-of-words_model) which is a sparse representation of text.

In essence, it's a way to compare how similar two pieces of text are based on the words they both contain.

This retriever is very straightforward to set-up! Let's see it happen down below!


In [71]:
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(loan_complaint_data, )

We'll construct the same chain - only changing the retriever.

In [72]:
bm25_retrieval_chain = (
    {"context": itemgetter("question") | bm25_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at the responses!

In [73]:
bm25_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'Based on the provided information, the most common issue with loans appears to involve problems with the handling and servicing of the loans by lenders or servicers. Specifically, the frequent issues include:\n\n- Dealing with lenders or servicers and disputes over fees charged\n- Trouble with how payments are applied, often leading to concerns about predatory practices\n- Receiving inaccurate or bad information about loan balances, interest calculations, or repayment terms\n- Issues related to loan justifications or reimbursement related to the validity of educational institutions\n\nOverall, the most common theme centers around problems with loan servicing, incorrect information, and complaints about how loan payments and fees are managed.'

In [74]:
bm25_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

"Based on the provided information, it appears that several complaints were responded to with a 'Company response to consumer' marked as 'Closed with explanation,' and the 'Timely response?' field indicates 'Yes' for those cases. Specifically, the complaints from rows 288, 423, and 13410623 all received responses within the expected timeframe. \n\nHowever, the most recent complaints, particularly the one received on 05/08/25, involve unresolved issues such as inadequate responses and ongoing disputes, but the documentation shows they were marked as 'Closed with explanation' and responded to on time.\n\nGiven this, there is no indication from the data that any complaints went unhandled or were not responded to in a timely manner. \n\nTherefore, the answer is: I do not know of any complaints that did not get handled in a timely manner based on the provided information."

In [75]:
bm25_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People failed to pay back their loans for various reasons, including issues with their repayment plans, miscommunication or lack of communication from loan servicers, errors with payment processing, and inability to access or receive appropriate information. Specific examples from the complaints indicate that some borrowers were steered into the wrong types of forbearances or had their autopayments unenrolled without their knowledge, leading to unpaid bills and negative impacts on their credit scores. Others experienced difficulties in obtaining responses or assistance from loan servicers when they tried to resolve issues or request deferments or forbearances. Overall, these problems often stem from procedural errors, inadequate communication, or alleged mismanagement by loan servicing companies.'

It's not clear that this is better or worse, if only we had a way to test this (SPOILERS: We do, the second half of the notebook will cover this)

#### ❓ Question #1:

Give an example query where BM25 is better than embeddings and justify your answer.

#### ✅ Answer #1:
BM25 would be better when looking for very specific and unique text such as serial numbers, model numbers, error codes, policy document names, etc. This is because BM25 ranks results based on exact matches. Embedding-based retrievers prioritize semantic similarity and may not find unique values, especially if they lack semantic meaning.

## Task 6: Contextual Compression (Using Reranking)

Contextual Compression is a fairly straightforward idea: We want to "compress" our retrieved context into just the most useful bits.

There are a few ways we can achieve this - but we're going to look at a specific example called reranking.

The basic idea here is this:

- We retrieve lots of documents that are very likely related to our query vector
- We "compress" those documents into a smaller set of *more* related documents using a reranking algorithm.

We'll be leveraging Cohere's Rerank model for our reranker today!

All we need to do is the following:

- Create a basic retriever
- Create a compressor (reranker, in this case)

That's it!

Let's see it in the code below!

In [76]:
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_cohere import CohereRerank

compressor = CohereRerank(model="rerank-v3.5")
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=naive_retriever
)

Let's create our chain again, and see how this does!

In [77]:
contextual_compression_retrieval_chain = (
    {"context": itemgetter("question") | compression_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [78]:
contextual_compression_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'Based on the provided information, a common issue with loans, particularly student loans, is dealing with errors, miscommunications, and mismanagement by lenders or servicers. This includes issues such as incorrect information about loan balances, misapplied payments, wrongful denials of payment plans, unauthorized transfers of loans, privacy violations, and mishandling of loan data. These problems often lead to disputes, damaged credit reports, and feelings of unclarity or mistrust regarding the loan process.'

In [79]:
contextual_compression_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided information, yes, there were complaints that did not get handled in a timely manner. For example, one complaint regarding a student loan issue has been open for over a year without resolution, and the complainant has indicated that it has been nearly 18 months with no response or resolution. Additionally, in the complaint submitted on 04/14/25, the consumer reports waiting over 1 year for a response and resolution to a request.'

In [80]:
contextual_compression_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People failed to pay back their loans mainly due to a combination of factors including lack of clear communication and understanding about their loans, unanticipated financial hardships, and issues with the management of their loans by servicers. Specifically, some borrowers were not adequately informed about their repayment obligations or the details of interest accumulation, leading to confusion and difficulty in making payments. Others experienced increased debt despite making payments, as interest continued to grow during deferment or forbearance periods, and options like forbearance extended the repayment timeline while adding to the total amount owed. Additionally, some borrowers faced challenges related to incorrect or inconsistent account information, lack of notification about payments or loan transfers, and difficulties in accessing accurate online account details. All these factors contributed to borrowers being unable to successfully pay back their loans.'

We'll need to rely on something like Ragas to help us get a better sense of how this is performing overall - but it "feels" better!

## Task 7: Multi-Query Retriever

Typically in RAG we have a single query - the one provided by the user.

What if we had....more than one query!

In essence, a Multi-Query Retriever works by:

1. Taking the original user query and creating `n` number of new user queries using an LLM.
2. Retrieving documents for each query.
3. Using all unique retrieved documents as context

So, how is it to set-up? Not bad! Let's see it down below!



In [81]:
from langchain.retrievers.multi_query import MultiQueryRetriever

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=naive_retriever, llm=chat_model
)

In [82]:
multi_query_retrieval_chain = (
    {"context": itemgetter("question") | multi_query_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [83]:
multi_query_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'Based on the provided complaints and data, the most common issue with loans appears to be problems related to "Dealing with your lender or servicer." Specifically, this includes issues such as miscommunication, errors in loan balances, misapplied payments, unauthorized transfers, lack of transparency, and problems with repayment or payment plans. Many consumers report difficulty in getting accurate information, handling errors, or resolving disputes with loan servicers. \n\nWhile other issues like incorrect information on reports or problems with forgiveness also occur, the predominant and recurring theme is dissatisfaction or conflict with loan servicers and the handling of loans.'

In [84]:
multi_query_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Yes, some complaints did not get handled in a timely manner. Notably, there are instances where responses from the companies were marked as "Not timely," and in several cases, consumers reported waiting over weeks or even months without resolution or response, despite making multiple follow-ups. For example:\n\n- Complaint ID 12709087 (submitted 03/28/25, sent to company the same day) was marked as "Response: No" but was handled with a "Closed with explanation" response from the company, indicating the issue was not resolved promptly.\n- Complaint ID 12654977 (submitted 03/25/25) was also marked as "Not timely," with the consumer reporting that their issues remained unresolved after over a year.\n- Several other complaints, such as ID 12823876 and ID 13056764, involved delays exceeding the law’s response times, with consumers waiting extensive periods and multiple follow-ups without adequate responses.\n\nOverall, the data indicates that although some responses were timely, a signific

In [85]:
multi_query_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People failed to pay back their loans primarily due to issues such as:\n\n1. Lack of proper information about interest accrual and repayment options, leading some to believe they could lower payments without increasing total debt.\n2. Being steered into long-term forbearances or consolidation without being informed of more beneficial options like income-driven repayment plans or loan rehabilitation, which can prevent interest from ballooning.\n3. Facing systemic mismanagement, errors, or misconduct by loan servicers, including incorrect reporting, failure to follow regulations, or mishandling of accounts (e.g., improper notices, bad information, or unauthorized disclosures).\n4. Experiencing financial hardships like unemployment, homelessness, illness, or unexpected personal crises, which made repayment unmanageable.\n5. Being misled or inadequately advised by servicers, resulting in increased debt via capitalization of interest or unfavorable repayment plans.\n6. Challenges in managi

#### ❓ Question #2:

Explain how generating multiple reformulations of a user query can improve recall.

#### 🎥✅ Answer #2:

User queries may not always be well formatted. Often, the hardest part of solving a problem is having a clear problem statement. For an analogy, think of an audience member asking a question of a presenter, and the presenter responding with: "I don't follow your question, could you state that another way, please". Having an LLM reformulate the questiion in different ways, increases the chances that the retriever will match documents that might have expressed the relevant idea with different wording.

## Task 8: Parent Document Retriever

A "small-to-big" strategy - the Parent Document Retriever works based on a simple strategy:

1. Each un-split "document" will be designated as a "parent document" (You could use larger chunks of document as well, but our data format allows us to consider the overall document as the parent chunk)
2. Store those "parent documents" in a memory store (not a VectorStore)
3. We will chunk each of those documents into smaller documents, and associate them with their respective parents, and store those in a VectorStore. We'll call those "child chunks".
4. When we query our Retriever, we will do a similarity search comparing our query vector to the "child chunks".
5. Instead of returning the "child chunks", we'll return their associated "parent chunks".

Okay, maybe that was a few steps - but the basic idea is this:

- Search for small documents
- Return big documents

The intuition is that we're likely to find the most relevant information by limiting the amount of semantic information that is encoded in each embedding vector - but we're likely to miss relevant surrounding context if we only use that information.

Let's start by creating our "parent documents" and defining a `RecursiveCharacterTextSplitter`.

In [86]:
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient, models

parent_docs = loan_complaint_data
child_splitter = RecursiveCharacterTextSplitter(chunk_size=750)

We'll need to set up a new QDrant vectorstore - and we'll use another useful pattern to do so!

> NOTE: We are manually defining our embedding dimension, you'll need to change this if you're using a different embedding model.

In [87]:
from langchain_qdrant import QdrantVectorStore

client = QdrantClient(location=":memory:")

client.create_collection(
    collection_name="full_documents",
    vectors_config=models.VectorParams(size=1536, distance=models.Distance.COSINE)
)

parent_document_vectorstore = QdrantVectorStore(
    collection_name="full_documents", embedding=OpenAIEmbeddings(model="text-embedding-3-small"), client=client
)

Now we can create our `InMemoryStore` that will hold our "parent documents" - and build our retriever!

In [88]:
store = InMemoryStore()

parent_document_retriever = ParentDocumentRetriever(
    vectorstore = parent_document_vectorstore,
    docstore=store,
    child_splitter=child_splitter,
)

By default, this is empty as we haven't added any documents - let's add some now!

In [89]:
parent_document_retriever.add_documents(parent_docs, ids=None)

We'll create the same chain we did before - but substitute our new `parent_document_retriever`.

In [90]:
parent_document_retrieval_chain = (
    {"context": itemgetter("question") | parent_document_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's give it a whirl!

In [91]:
parent_document_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'The most common issue with loans, based on the provided context, appears to be "Federal student loan servicing" issues, which include problems such as incorrect information on credit reports, misconduct by servicers (errors in loan balances, misapplied payments, wrongful denials of payment plans), disputes over interest rates, and issues related to loan balance verification and credit reporting. Many complaints involve systemic breakdowns, miscommunication, or errors by loan servicers that negatively impact borrowers\' credit and financial stability.'

In [92]:
parent_document_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided data, several complaints have been marked as "Timely response?": "No," indicating they were not handled in a timely manner. Specifically, complaints with IDs 12709087 and 12935889 from Mohela, received on 03/28/25 and 04/11/25 respectively, are noted as not being responded to promptly. These complaints involved delays in follow-up and communication from the company.\n\nTherefore, yes, there were complaints that did not get handled in a timely manner.'

In [93]:
parent_document_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'Based on the provided complaints, people failed to pay back their loans primarily due to a combination of factors such as financial hardship, mismanagement, lack of proper communication, and unforeseen circumstances. For example:\n\n- Some borrowers experienced severe financial hardship after graduation and relied on deferment or forbearance, which increased interest and made repayment more difficult.\n- Others were misinformed or not properly advised about the long-term consequences of taking on student loans, and faced issues with their loan servicing agencies, such as payments being expected before the grace period ended or loans being reported as delinquent without proper notification.\n- In certain cases, the schools the borrowers attended closed or faced financial problems, leaving students without the expected job prospects or education value, which hindered their ability to repay the loans.\n- Additional issues involved administrative errors, such as failure to verify debt leg

Overall, the performance *seems* largely the same. We can leverage a tool like [Ragas]() to more effectively answer the question about the performance.

## Task 9: Ensemble Retriever

In brief, an Ensemble Retriever simply takes 2, or more, retrievers and combines their retrieved documents based on a rank-fusion algorithm.

In this case - we're using the [Reciprocal Rank Fusion](https://plg.uwaterloo.ca/~gvcormac/cormacksigir09-rrf.pdf) algorithm.

Setting it up is as easy as providing a list of our desired retrievers - and the weights for each retriever.

In [94]:
from langchain.retrievers import EnsembleRetriever

retriever_list = [bm25_retriever, naive_retriever, parent_document_retriever, compression_retriever, multi_query_retriever]
equal_weighting = [1/len(retriever_list)] * len(retriever_list)

ensemble_retriever = EnsembleRetriever(
    retrievers=retriever_list, weights=equal_weighting
)

We'll pack *all* of these retrievers together in an ensemble.

In [95]:
ensemble_retrieval_chain = (
    {"context": itemgetter("question") | ensemble_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at our results!

In [96]:
ensemble_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'Based on the provided complaints data, the most common issues with loans are related to dealing with lenders or servicers, particularly issues such as receiving bad information, trouble with how payments are handled, problems with loan transfer or servicing company changes, and disputes over loan balances and interest calculations. Many complaints highlight a lack of transparency, improper handling, or documentation problems.\n\nTherefore, the most common issue with loans appears to be **"Dealing with your lender or servicer,"** which encompasses various sub-issues such as inaccurate information, mishandling of payments, loan transfer concerns, and inadequate communication or documentation.\n\nIf you have a specific aspect or a different focus in mind, please let me know!'

In [97]:
ensemble_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided complaints, yes, several complaints indicate that issues were not handled in a timely manner. For example:\n\n- A complaint from 04/18/25 about a complaint being marked as "Closed with explanation" when the consumer insists it was not resolved, and the response claims the issue was handled appropriately, but the consumer disagrees.\n- A complaint from 04/01/25 shows they did not respond within the 30-day period required for investigations.\n- Multiple complaints mention delays, unresponsive customer service, or unresolved issues spanning over weeks or months.\n\nIn particular, the complaint with ID 12935889 from 04/11/25 indicates a response that was "Not timely," and complaints about delays in correcting credit report inaccuracies or handling account issues also highlight delays in handling issues.\n\nWhile some complaints received responses marked as "timely" or "closed with explanation," the recurring theme across several complaints suggests that many issues w

In [98]:
ensemble_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People often fail to pay back their loans due to a variety of complex and interconnected reasons, including:\n\n1. Lack of clear information: Borrowers are frequently misled or inadequately informed about the true costs of their loans, including how interest compounds and the implications of forbearance or deferment options. Many are unaware of additional interest accruing during payment pauses, which can significantly increase the total amount owed.\n\n2. Economic hardships: Borrowers face financial hardships such as unemployment, medical emergencies, homelessness, or reduced income due to economic downturns or personal circumstances, making it difficult to meet repayment obligations.\n\n3. Unsuitable repayment options: Limited or unhelpful payment plans, such as only being offered forbearance or deferment, often lead to ongoing interest accumulation. When available options like income-driven repayment plans are not properly communicated or accessible, borrowers struggle to manage pa

## Task 10: Semantic Chunking

While this is not a retrieval method - it *is* an effective way of increasing retrieval performance on corpora that have clean semantic breaks in them.

Essentially, Semantic Chunking is implemented by:

1. Embedding all sentences in the corpus.
2. Combining or splitting sequences of sentences based on their semantic similarity based on a number of [possible thresholding methods](https://python.langchain.com/docs/how_to/semantic-chunker/):
  - `percentile`
  - `standard_deviation`
  - `interquartile`
  - `gradient`
3. Each sequence of related sentences is kept as a document!

Let's see how to implement this!

We'll use the `percentile` thresholding method for this example which will:

Calculate all distances between sentences, and then break apart sequences of setences that exceed a given percentile among all distances.

In [99]:
from langchain_experimental.text_splitter import SemanticChunker

semantic_chunker = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile"
)

Now we can split our documents.

In [100]:
semantic_documents = semantic_chunker.split_documents(loan_complaint_data[:20])

Let's create a new vector store.

In [101]:
semantic_vectorstore = Qdrant.from_documents(
    semantic_documents,
    embeddings,
    location=":memory:",
    collection_name="Loan_Complaint_Data_Semantic_Chunks"
)

We'll use naive retrieval for this example.

In [102]:
semantic_retriever = semantic_vectorstore.as_retriever(search_kwargs={"k" : 10})

Finally we can create our classic chain!

In [103]:
semantic_retrieval_chain = (
    {"context": itemgetter("question") | semantic_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

And view the results!

In [104]:
semantic_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'Based on the provided context, the most common issues with loans appear to relate to problems with loan servicing and administration. These include struggles to repay loans due to unclear or incorrect payment plans, mismanagement or delays in processing repayment applications, inaccurate reporting on credit reports, difficulties in understanding or obtaining loan information, and issues with loan discharges or forgiveness. Many complaints highlight poor communication, lack of transparency, and improper handling of borrower data.\n\nTherefore, it seems that the most common issue with loans, especially student loans in this context, is **administrative and servicing problems**, such as mismanagement, incorrect information, or delays in processing repayment or discharge requests.\n\nIf you have a specific aspect or type of issue in mind, please let me know!'

In [105]:
semantic_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Yes, based on the provided complaints, several complaints indicate that they were handled in a timely manner. All complaints explicitly state "Timely response?": "Yes". \n\nHowever, since the question asks if any complaints did **not** get handled in a timely manner, and there is no explicit record of complaints being handled late or improperly, the data does not show any complaints that were not responded to within the expected timeframe. \n\nTherefore, the answer is:  \n**No, there is no evidence from the provided data that any complaints did not get handled in a timely manner.**'

In [106]:
semantic_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

"People failed to pay back their loans for various reasons, including issues with loan servicing, miscommunication, technical problems, or legal disputes. For example, some borrowers experienced difficulties due to poor communication or lack of transparency from their lenders or servicers, such as receiving little to no contact or bad information about their loan status. Others faced challenges because of administrative errors, such as missing payments or discrepancies in payment records, which made it difficult to verify their payments and maintain accurate account status. In some cases, borrowers encountered delays or stalls in processing documents needed for loan forgiveness or discharge, leading them to give up or become overwhelmed. Additionally, legal and privacy concerns, such as data breaches or disputes over the legality of certain debts, contributed to borrowers' inability to resolve or repay their loans effectively."

#### ❓ Question #3:

If sentences are short and highly repetitive (e.g., FAQs), how might semantic chunking behave, and how would you adjust the algorithm?

#### ✅ Answer #3:

Short answer is that semantic chunking in this case would group many of the short, similar sentences together. And, this would probably be desirable. But, if the bar is too low, it might group together sentences that are only superficially similar. To adjust this, you would raise the threshold, e.g. in this case, the percentile value that triggers a split.

I also wanted to make sure I understood the math and process behind semantic chunking. 


The Process
First, split document into sentences (langchain uses sentence-based splitter, e.g. nltk.sent_tokenizer)
Then do pairwise comparisons of the semantic distance between sentences.
Then group sentences if they are "close" in semantic meaning, and split to a new chunk when the semantic distance to the previous sentence exceeds the threshold. 

The Math

Percentile: 
Split when the semantic distance between two adjacent sentences is bigger than the n-th percentile of all adjacent-pair distances in the document. (Typical starting threshold: **75th or 80th percentile**. Lower the threshold to get smaller chunks (e.g. if topic shifts frequently) or raise the percentil to get fewer larger chunks)

Standard Deviation: 
Split when the semantic distance between two adjacent sentences exceeds the mean by nore than n times the stddev of all adjacent-pair distances in the document. (Typical value: **n = 1.5**; threshold = **Mean + 1.5 × StdDev**)

Interquartile:
Split when the distance falls above the Q3 percentile by some factor, "n", of the "interquartile range" (IQR). IQR is the Q3 percentile - Q1 percentile. Example: threshold = Q3 + n x IQR (Typical value: **n = 1.5**; threshold = **Q3 + 1.5 × IQR**)

Gradient:
This is kind of like a first order derivative, compared to the others. It essentially measures the velocity of change between the sentences (as opposed ot the distance), triggering a split when there is a sudden change. To do this, after calculating the distances between each set of pairs, you then calculate the gradients (the distances between the distances). The rule would be to split whenever the gradient exceeds a threshold, **N**.  
(Typical value: **N = 0.10**)

🎥 See example [here](semantic_chunking_examples.md) that I worked out with ChatGPT



# 🤝 Breakout Room Part #2

#### 🏗️ Activity #1

Your task is to evaluate the various Retriever methods against eachother.

You are expected to:

1. Create a "golden dataset"
 - Use Synthetic Data Generation (powered by Ragas, or otherwise) to create this dataset
2. Evaluate each retriever with *retriever specific* Ragas metrics
 - Semantic Chunking is not considered a retriever method and will not be required for marks, but you may find it useful to do a "semantic chunking on" vs. "semantic chunking off" comparision between them
3. Compile these in a list and write a small paragraph about which is best for this particular data and why.

Your analysis should factor in:
  - Cost
  - Latency
  - Performance

> NOTE: This is **NOT** required to be completed in class. Please spend time in your breakout rooms creating a plan before moving on to writing code.

##### HINTS:

- LangSmith provides detailed information about latency and cost.

In [107]:
### YOUR CODE HERE





#### Study Note
these are my notes taken during breakout

generate golde dataest
naive, bm25, reranking, etc.
should find all the code from 8
import ragas llms from 7 
#uv add ragas 
#nltk?
from raqas.llms import Langchainllmwrapper 

#### Study Note: Documentation
I'm going to document the heck out of this.
If I track every step that I do, I'll know what to say in my Loom video 🤞
First off, I'm eager and nervous to try this.
I'm pretty sure I can find all the code I need in notebooks from session 7 & 8

#### Study Note: Design Idea
Activity 1 instructions seem pretty clear, but I ran it by ChatGPT to confirm.
I need to 
1. Create synth dataset for test case
2. Evaluate 5 retreival approaches: Naive, BM25, Multi-Query, Parent Doc, Ensemble with Ragas
3. Then I'll have to do something with LangSmith to get the latency and cost, but I'll think about that later



## Task 1: Create a dataset

I will find the ragas code from notebook 7

First, I'll do the nltk thing. I still don't undertsand it, but the comment in 7 said it would prevent (mac related) os errors.
It worked there, so I will keep it.
First stumbling block: it failed. But, I figured out why pretty quick. This notebook didn't install nltk

#### Study Note: Environment setup
Till now, I've just been relying on uv sync, and not thinking much about dependencies.
So, I took a look at the project.toml for this notebook vs 7. Lots of stuff in 7 that isn't here, and I think I'm going to need it.
But, I don't want to break anything, so, I'll just add things as I run into needing them (starting with nltk). Also, I'll just install via terminal first, and if nothing breaks, I'll update the toml file later.
Clearly, I'm also going to need ragas
Ran this command:  uv pip install nltk ragas==0.2.10



#### Study Notes: To-dos
Track thinkgs here that are pending
update toml with nltk and ragas (see above)
update toml with rapidfuzz (see below)

In [108]:
import nltk
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package punkt to /Users/family/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/family/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


True

That worked. Baby steps!

I'm going to end up doing something with LangSmith, so I'll steal the next cell from earlier notebooks (plus a print statement)

In [116]:
from uuid import uuid4

os.environ["LANGCHAIN_PROJECT"] = f"Number9 - {uuid4().hex[0:8]}"

print(f"Project name: {os.environ['LANGCHAIN_PROJECT']}")

Project name: Number9 - 5279ee98


Now I'll import stuff and set the llm 
I don't understand the last line, but not going to worry about it right now 

In [110]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-nano"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

Taking the 'abstracted SDG' from 7
i rant it as it was in 7, and I got an error, because there was no such thing as "docs", so i figured out that I needed to put in "loan_complaint_data"
pretty pleased that I figured that out, with only a slight nudge from ChatGPT
It failed, with a nice clear error message that I needed to install "rapidfuzz".

Don't know why I need rapidfuzz now, when we didn't need it in 7. 
I suppose it is because of the structure of the data.
ChatGPT gave me some other hogwash that I don't believe. I'm sticking with my guess, and it doesn't really matter.
I installed rapidfuzz (in terminal window; I'll need to add a cell or put it in the toml file later)

In [112]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
dataset = generator.generate_with_langchain_docs(loan_complaint_data[:20], testset_size=10)

Applying SummaryExtractor:   0%|          | 0/14 [00:00<?, ?it/s]

Applying CustomNodeFilter:   0%|          | 0/20 [00:00<?, ?it/s]

Node e52eb778-c7a8-47a2-8a71-08a7c11c94c7 does not have a summary. Skipping filtering.
Node cdc7c363-9cba-406b-a4b2-4645b62b2a43 does not have a summary. Skipping filtering.
Node d8eb5680-f416-46ac-83a7-7423945f6160 does not have a summary. Skipping filtering.
Node 4e6c8469-4508-4c03-be55-2bcad65b68fc does not have a summary. Skipping filtering.
Node 4376a08f-552f-4843-bb89-433cf3381694 does not have a summary. Skipping filtering.
Node 085cf9ed-128e-4f5e-a0e5-bb0b5d8019fd does not have a summary. Skipping filtering.


Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/54 [00:00<?, ?it/s]

Applying OverlapScoreBuilder:   0%|          | 0/1 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/2 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/10 [00:00<?, ?it/s]

Wow, that worked!
Now, I need to see the dataset, so I'm stealing the next cell from notebook 7

I am gobsmacked that I got this working in under an hour (not counting annotations) Insert excessive emojis here: ✅❤️🥳🎉
Time to (commit the code), take a victory lap, and call it a night.

In [113]:
dataset.to_pandas()

,user_input,reference_contexts,reference,synthesizer_name
0,How did the COVD-19 forbearance program impact...,[The federal student loan COVID-19 forbearance...,The federal student loan COVID-19 forbearance ...,single_hop_specifc_query_synthesizer
1,"As a Student Borrower Advocate, can you explai...",[I submitted my annual Income-Driven Repayment...,Aidvantage assigned the borrower a repayment a...,single_hop_specifc_query_synthesizer
2,How FERPA was violated in my data breach?,[My personal and financial data was compromise...,My personal and financial data was compromised...,single_hop_specifc_query_synthesizer
3,According to the information on Studentaid.gov...,"[According to Studentaid.gov, Im to get an ema...",The context indicates that Studentaid.gov stat...,single_hop_specifc_query_synthesizer
4,What does 15 U.S.C. 1681i require credit repor...,[I am writing to formally dispute inaccurate i...,"Under 15 U.S.C. 1681i, credit reporting agenci...",single_hop_specifc_query_synthesizer
5,How does the illegal reporting and collection ...,[<1-hop>\n\nIllegal Student Loan Reporting & C...,The illegal reporting and collection of studen...,multi_hop_specific_query_synthesizer
6,Based on the issues with NelNet's processing o...,[<1-hop>\n\nXX/XX/XXXX I increased the amount ...,The borrower’s experience demonstrates that di...,multi_hop_specific_query_synthesizer
7,"So, if I dispute wrong info on my credit repor...",[<1-hop>\n\nI am writing to formally dispute i...,"The context shows that under the FCRA, credit ...",multi_hop_specific_query_synthesizer
8,Based on the violations and misconduct detaile...,[<1-hop>\n\nThis account was transferred to Ne...,The account transferred to Nelnet involved act...,multi_hop_specific_query_synthesizer
9,How did Nelnet's handling of the federal stude...,[<1-hop>\n\nBreach of Contract - All four bran...,"The context indicates that Nelnet, as the loan...",multi_hop_specific_query_synthesizer
